Procesado de datos de las baterías 13300 para el TFG de reciclado de baterías de vape.

Primero analizaremos los ciclos y veremos el SoH de las baterías

In [26]:
import os
import shutil
import re
import glob
from pathlib import Path
import numpy as np
from bateria import Bateria
import csv

def organizar_archivos_por_fecha(carpeta_origen):
    """
    Lee los archivos de una carpeta, extrae la fecha de su nombre 
    y los mueve a subcarpetas organizadas por día.
    """
    
    # Expresión regular para buscar el formato de fecha AAAA_MM_DD
    # \d{4} = 4 dígitos (año), \d{2} = 2 dígitos (mes/día)
    patron_fecha = re.compile(r"(\d{4}_\d{2}_\d{2})")

    # Verificamos que la carpeta de origen exista
    if not os.path.exists(carpeta_origen):
        print(f"Error: La carpeta '{carpeta_origen}' no existe.")
        return

    # Iterar sobre todos los elementos en la carpeta
    for nombre_archivo in os.listdir(carpeta_origen):
        ruta_completa_origen = os.path.join(carpeta_origen, nombre_archivo)

        # Nos aseguramos de procesar solo archivos (ignoramos carpetas)
        if os.path.isfile(ruta_completa_origen):
            
            # Buscamos el patrón de la fecha en el nombre del archivo
            coincidencia = patron_fecha.search(nombre_archivo)
            
            if coincidencia:
                # Extraemos la fecha encontrada (ej. '2026_02_23')
                fecha_carpeta = coincidencia.group(1) 
                
                # Definimos la ruta de la nueva carpeta destino
                ruta_carpeta_destino = os.path.join("data/processed/cicladores", str(i), fecha_carpeta)
                
                # Si la carpeta de esa fecha no existe, la creamos
                if not os.path.exists(ruta_carpeta_destino):
                    os.makedirs(ruta_carpeta_destino)
                
                # Movemos el archivo
                ruta_completa_destino = os.path.join(ruta_carpeta_destino, nombre_archivo)
                shutil.copy(ruta_completa_origen, ruta_completa_destino)
                
                #print(f"Copiado: {nombre_archivo}  ->  Carpeta: {fecha_carpeta}/")
            else:
                print(f"Omitido (sin fecha detectada): {nombre_archivo}")



In [27]:
def trim_txt_files(processed_dir: str = "data/processed/cicladores") -> dict:
    """
    For every .txt file under data/processed/{1-6}/{dates}/:
      - Overwrites each file keeping only the header (line 1) and the last data row.
    For every .dat file in the same folders:
      - Deletes the file entirely.

    Parameters
    ----------
    processed_dir : str
        Path to the 'processed' folder, relative to the notebook location.
        Defaults to "data/processed".

    Returns
    -------
    dict with keys:
        "processed"  – list of .txt files successfully trimmed
        "skipped"    – list of .txt files skipped (< 2 lines or already trimmed)
        "deleted"    – list of .dat files deleted
        "errors"     – list of (filepath, error_message) tuples
    """
    base = Path(processed_dir)
    results = {"processed": [], "skipped": [], "deleted": [], "errors": []}

    def is_valid_folder(p: Path) -> bool:
        return p.parts[-3].isdigit() and 1 <= int(p.parts[-3]) <= 6

    # --- Process .txt files ---
    for filepath in sorted(base.glob("*/*/*.txt")):
        if not is_valid_folder(filepath):
            continue
        try:
            lines = filepath.read_text(encoding="latin-1").splitlines()

            if len(lines) < 2 or len(lines) == 2:
                results["skipped"].append(str(filepath))
                continue

            header   = lines[0]
            last_row = lines[-1]

            filepath.write_text(header + "\n" + last_row + "\n", encoding="latin-1")
            results["processed"].append(str(filepath))

        except Exception as e:
            results["errors"].append((str(filepath), str(e)))

    # --- Delete .dat files ---
    for filepath in sorted(base.glob("*/*/*.dat")):
        if not is_valid_folder(filepath):
            continue
        try:
            filepath.unlink()
            results["deleted"].append(str(filepath))
        except Exception as e:
            results["errors"].append((str(filepath), str(e)))

    # Summary
    print(f"✅ Trimmed : {len(results['processed'])} .txt files")
    print(f"⏭️  Skipped : {len(results['skipped'])} .txt files (already 2 lines or empty)")
    print(f"🗑️  Deleted : {len(results['deleted'])} .dat files")
    print(f"❌ Errors  : {len(results['errors'])} files")
    if results["errors"]:
        for fp, err in results["errors"]:
            print(f"   {fp}: {err}")

    return results

In [28]:
for i in range(1, 7):

    mi_carpeta = f"data/raw/cicladores/ciclador_{i}"
    
    print(f"Procesando: {mi_carpeta} ...")
    
    # Llamamos a la función con la ruta actualizada
    organizar_archivos_por_fecha(mi_carpeta)

print("¡Todas las carpetas han sido organizadas!")

Procesando: data/raw/cicladores/ciclador_1 ...
Procesando: data/raw/cicladores/ciclador_2 ...
Procesando: data/raw/cicladores/ciclador_3 ...
Procesando: data/raw/cicladores/ciclador_4 ...
Procesando: data/raw/cicladores/ciclador_5 ...
Procesando: data/raw/cicladores/ciclador_6 ...
¡Todas las carpetas han sido organizadas!


In [29]:
results = trim_txt_files() #porque ya conoce que tiene que buscar en data/processed

✅ Trimmed : 129 .txt files
⏭️  Skipped : 21 .txt files (already 2 lines or empty)
🗑️  Deleted : 150 .dat files
❌ Errors  : 0 files


In [30]:
"""
class Bateria:
    def __init__(self, numero: int, soh: float, impedancia: float):
        self.numero = numero        # Identificador de la batería
        self.soh = soh              # State of Health (%) — rango 0.0 a 100.0
        self.impedancia = impedancia  # Impedancia interna (Ohmios)

    def __repr__(self):
        return (f"Bateria(numero={self.numero}, "
                f"soh={self.soh}%, "
                f"impedancia={self.impedancia}Ω)")

    def esta_en_buen_estado(self) -> bool:
        return self.soh >= 0.3460*0.8

    def resumen(self) -> str:
        estado = "✓ Buena" if self.esta_en_buen_estado() else "✗ Degradada"
        return (f"Batería #{self.numero} — Estado: {estado} | "
                f"SoH: {self.soh}% | Impedancia: {self.impedancia}Ω")

def build_soh_matrix(baterias_dir: str = "data/processed/baterias") -> np.ndarray:
    
    Iterates over all battery folders in data/processed/baterias/ and extracts
    the SOH PS (Ah) value from the single data row of each .txt file.

    Expected file structure:
        baterias/{battery_number}/{file}.txt
    Each .txt has exactly 2 lines: a header row and one data row (pre-trimmed).

    Parameters
    ----------
    baterias_dir : str
        Path to the 'baterias' folder, relative to the notebook location.

    Returns
    -------
    np.ndarray
        1 x N matrix (shape (1, N)) with the SOH PS (Ah) value per battery folder,
        sorted by folder name. Folders with missing/invalid files get np.nan.
    
    base = Path(baterias_dir)

    # Grab all subdirectories, sorted so the order is deterministic
    battery_folders = sorted([p for p in base.iterdir() if p.is_dir()], key=lambda p: int(p.name))
    if not battery_folders:
        print(f"No folders found under '{base}'. Check the path.")
        return np.empty((1, 0))

    soh_values = []

    for folder in battery_folders:
        txt_files = list(folder.glob("*.txt"))

        if not txt_files:
            print(f"⚠️  No .txt file found in '{folder.name}' — inserting NaN.")
            soh_values.append(np.nan)
            continue

        if len(txt_files) > 1:
            print(f"⚠️  Multiple .txt files in '{folder.name}' — using first: {txt_files[0].name}")

        filepath = txt_files[0]

        try:
            lines = filepath.read_text(encoding="latin-1").splitlines()

            # Find the SOH PS (Ah) column index from the header
            header_cols = [col.strip() for col in lines[0].split(",")]
            soh_col_idx = header_cols.index("SOH PS (Ah)")

            # Extract value from the data row
            data_cols = lines[1].split(",")
            soh_value = float(data_cols[soh_col_idx])
            soh_values.append(soh_value)

        except ValueError as e:
            print(f"⚠️  Could not parse '{filepath}': {e} — inserting NaN.")
            soh_values.append(np.nan)
        except Exception as e:
            print(f"❌  Unexpected error with '{filepath}': {e} — inserting NaN.")
            soh_values.append(np.nan)

    print(f"✅ Built SOH matrix of shape {len(soh_values)} from {len(battery_folders)} battery folders.")

    return soh_values
"""

'\nclass Bateria:\n    def __init__(self, numero: int, soh: float, impedancia: float):\n        self.numero = numero        # Identificador de la batería\n        self.soh = soh              # State of Health (%) — rango 0.0 a 100.0\n        self.impedancia = impedancia  # Impedancia interna (Ohmios)\n\n    def __repr__(self):\n        return (f"Bateria(numero={self.numero}, "\n                f"soh={self.soh}%, "\n                f"impedancia={self.impedancia}Ω)")\n\n    def esta_en_buen_estado(self) -> bool:\n        return self.soh >= 0.3460*0.8\n\n    def resumen(self) -> str:\n        estado = "✓ Buena" if self.esta_en_buen_estado() else "✗ Degradada"\n        return (f"Batería #{self.numero} — Estado: {estado} | "\n                f"SoH: {self.soh}% | Impedancia: {self.impedancia}Ω")\n\ndef build_soh_matrix(baterias_dir: str = "data/processed/baterias") -> np.ndarray:\n\n    Iterates over all battery folders in data/processed/baterias/ and extracts\n    the SOH PS (Ah) value from

Hemos limpiado el sistema de archivos: El resultado en la carpeta processed es una serie de archivos .txt en el que solamente se encuentra el último estado del ciclo que corresponde. Lo que nos interesa a continuación es casar cada archivo txt de Charge con la última carga correspondiente a cada batería.

In [31]:


def build_soh_matrix(baterias_dir: str = "data/processed/baterias", output_csv: str = "data/processed/matriz_baterias.csv"):
    base = Path(baterias_dir)
    # Listar y ordenar carpetas
    battery_folders = sorted([p for p in base.iterdir() if p.is_dir()], key=lambda p: int(p.name))

    if not battery_folders:
        print(f"No folders found under '{base}'. Check the path.")
        return []

    baterias = []

    for folder in battery_folders:
        numero    = int(folder.name)
        txt_files = list(folder.glob("*.txt"))

        if not txt_files:
            print(f"No .txt file found in '{folder.name}' — inserting None.")
            baterias.append(None)
            continue

        if len(txt_files) > 1:
            print(f"Multiple .txt files in '{folder.name}' — using first: {txt_files[0].name}")

        try:
            lines       = txt_files[0].read_text(encoding="latin-1").splitlines()
            header_cols = [col.strip() for col in lines[0].split(",")]
            soh_value   = float(lines[1].split(",")[header_cols.index("SOH PS (Ah)")])
            
            baterias.append(Bateria(numero=numero, soh=soh_value, impedancia=0))

        except Exception as e:
            print(f"Error with '{folder.name}': {e} — inserting None.")
            baterias.append(None)

    # --- NUEVO: Grabación de la matriz en CSV ---
    try:
        with open(output_csv, mode="w", newline="", encoding="utf-8") as csv_file:
            writer = csv.writer(csv_file)
            # Escribir el encabezado
            writer.writerow(["Numero", "SoH(Ah)", "Impedancia"])
            
            # Escribir los datos iterando simultáneamente sobre las carpetas y los objetos
            for folder, bat in zip(battery_folders, baterias):
                if bat is None:
                    # Si hubo error, guardamos el número de la carpeta y dejamos en blanco lo demás
                    writer.writerow([int(folder.name), "", ""])
                else:
                    writer.writerow([bat.numero, bat.soh, bat.impedancia])
                    
        print(f"\nMatriz guardada exitosamente en: '{output_csv}'")
    except Exception as e:
        print(f"\nError al guardar el archivo CSV: {e}")

    return baterias

In [32]:
def find_best_18(baterias: list) -> list:
    values         = np.array([b.soh for b in baterias])
    sorted_indices = np.argsort(values)
    sorted_values  = values[sorted_indices]

    best_window, best_spread, best_mean = 0, np.inf, -np.inf

    for i in range(len(values) - 17):
        window = sorted_values[i : i + 18]
        spread, mean = window[-1] - window[0], window.mean()
        if spread < best_spread or (spread == best_spread and mean > best_mean):
            best_window, best_spread, best_mean = i, spread, mean

    selected = sorted_indices[best_window : best_window + 18]
    return [baterias[i] for i in sorted(selected)]

Con esto, ya hemos encontrado las 18 baterías más compatibles para formar un sólo pack. Ahora, solo queda montarlo y ensayarlo.

In [33]:
baterias = build_soh_matrix()

best_baterias = find_best_18(baterias)

print(best_baterias)


Matriz guardada exitosamente en: 'data/processed/matriz_baterias.csv'
[Bateria(numero=1, soh=0.335735, impedancia=0Ω), Bateria(numero=2, soh=0.340149, impedancia=0Ω), Bateria(numero=6, soh=0.345978, impedancia=0Ω), Bateria(numero=7, soh=0.339422, impedancia=0Ω), Bateria(numero=9, soh=0.342146, impedancia=0Ω), Bateria(numero=10, soh=0.340374, impedancia=0Ω), Bateria(numero=12, soh=0.335763, impedancia=0Ω), Bateria(numero=13, soh=0.343665, impedancia=0Ω), Bateria(numero=15, soh=0.346292, impedancia=0Ω), Bateria(numero=17, soh=0.333173, impedancia=0Ω), Bateria(numero=20, soh=0.344466, impedancia=0Ω), Bateria(numero=21, soh=0.345323, impedancia=0Ω), Bateria(numero=24, soh=0.334309, impedancia=0Ω), Bateria(numero=26, soh=0.344176, impedancia=0Ω), Bateria(numero=29, soh=0.337994, impedancia=0Ω), Bateria(numero=33, soh=0.334904, impedancia=0Ω), Bateria(numero=35, soh=0.340035, impedancia=0Ω), Bateria(numero=36, soh=0.339868, impedancia=0Ω)]
